In [1]:
!ls stand

attempt			 games_dict		 meta.json
bin			 games_flog.gz		 order_money_left_caesar
config			 games_flog.gz.PREV	 poor_mans_profiler
daemon.json		 geodata4-tree+ling.bin  push-client
dssm			 geodata6.bin		 secret
dumped_requests.gz	 language		 service_tickets
dumped_requests.gz.PREV  lm			 stand.json
formulas		 log.local.txt		 unified_agent


In [2]:
!head stand/log.local.txt

Model = 6; [-0.211147,0.028769,-0.242279,0.0841654,0.0248732,0.0154993,0.0717655,0.0942284,-0.00700669,-0.0135621,0.0214947,0.022302,0.252389,-0.0336773,-0.138279,-0.0534052,0.0690308,-0.197792,-0.109553,-0.00141407,0.0443739,0.172978,-0.00064369,0.0590741,-0.0824093,-0.0682868,0.0344061,0.0911272,-0.0051644,-0.0839934,0.00860636,-0.0956932,-0.0486178,-0.0878616,-0.00121456,-0.0839231,-0.0282804,0.00692664,0.102235,-0.0945008,0.0576525,0.0920146,0.140158,-0.0620869,-0.20815,0.00623474,0.105515,0.215129,0.0282423,-0.0290657,-0.0268382,0.0365566,-0.0845465,0.0747499,-0.0457398,-0.0335554,-0.0625323,-0.11611,-0.00558053,0.0719073,-0.114699,0.0206136,-0.0253669,-0.0439999,0.326458,-0.0629797,-0.0288919,-0.0863563,-0.0421662,-0.0153366,-0.0748693,-0.0997181,-0.0855427,-0.105331,0.0707908,0.0398544,0.0028728,0.155376,0.138392,-0.00578193,-0.00382951,0.0407524,0.090671,0.267034,0.0891853,0.00231596,0.159994,0.105253,0.149872,0.00413346,0.0340003,-0.109571,0.0123925,0.0624056,-0.0539893,-0.031

In [56]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [76]:
import collections
import numpy as np
import tqdm

readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
requests = list()
docembs = collections.defaultdict(dict)
with open("stand/log.local.txt") as f:
    req = list()
    reqid = None
    models = list()
    prevreqmodel = None
    reqmodel = dict()
    prevmodelid = None
    for i, line in tqdm.tqdm_notebook(enumerate(f)):
        if line.startswith("Model = 6;"):
            prevreqmodel = reqmodel
            reqmodel = dict()

        if line.startswith("Model = "):
            spl = line.split(" ")
            prevmodelid = int(spl[2][:-1])
            reqmodel[prevmodelid] = readvector(spl[3])
        elif line.startswith("dbid"):
            spl = line.split(" ")
            dbid = int(spl[1][:-1])
            docembs[prevmodelid][dbid] = readvector(spl[2])
        elif line.startswith("seed"):
            if len(requests) >= 1000:
                break
            if req:
                requests.append((reqid, prevreqmodel, sorted(req)))
                req = list()
            reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
        else:
            req.append(
                (int(line.split()[0]), float(line.split()[1]))
            )

requests = [r for r in requests if len(r[2]) == 16514]

In [77]:
np.unique([len(requests_i[2]) for requests_i in requests], return_counts=True)

(array([16514]), array([987]))

In [113]:
docblocks = {
    mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
    for mid in docembs
}
docblocks

{6: array([[ 0.0407761 , -0.0866397 ,  0.0973855 , ..., -0.0337863 ,
          0.1449    ,  0.023338  ],
        [-0.0173765 ,  0.103833  ,  0.0541691 , ...,  0.0947965 ,
          0.119532  ,  0.111093  ],
        [-0.010643  , -0.0188779 ,  0.128833  , ..., -0.0185986 ,
          0.0270978 ,  0.0370004 ],
        ...,
        [-0.0822528 ,  0.0734386 ,  0.0820323 , ..., -0.0283938 ,
          0.0727261 ,  0.244681  ],
        [-0.0396752 ,  0.215461  ,  0.148155  , ..., -0.0594413 ,
          0.0757048 ,  0.140465  ],
        [-0.0272712 ,  0.207217  ,  0.154168  , ..., -0.00856583,
          0.0459061 ,  0.18192   ]]),
 7: array([[-0.0457229 ,  0.0874625 ,  0.0102194 , ...,  0.0135331 ,
         -0.0444551 ,  0.012255  ],
        [-0.135033  ,  0.0949462 , -0.0646434 , ...,  0.105708  ,
         -0.0441772 ,  0.0313092 ],
        [-0.176407  ,  0.15634   , -0.0423812 , ...,  0.165932  ,
         -0.0977578 , -0.0507538 ],
        ...,
        [ 0.0745855 ,  0.15481   , -0.0633103 , 

In [138]:
(requests[0][1][6] ** 2).sum()

0.9994653378993866

In [79]:
games_count = 16514
key_games = np.random.choice(np.arange(games_count), 100, replace=False)
key_reqs = np.random.choice(np.arange(len(requests)), 101, replace=False)
key_games, key_reqs

(array([12856, 16478,  2379,  6166, 15513, 14221, 13751, 12447,  4102,
        11673,  7311,   188,  9236,  9060,  9617, 10315,  6149, 10421,
         6505, 16181, 14141,  9370,  7281, 15254,  2514, 13665,  1756,
         5769,  8110,  9546, 13161, 12268,  4139, 15838,  4762, 12260,
         2220,  1770,  3538, 13215, 14670,  1979, 16297,  4052, 15148,
         4572, 14392,  3508, 14049,  1231,  6955,  8896, 16318, 12986,
         8186,  1167,  8592,  8878,  8901, 15406,  3362,  7040, 13035,
        12713, 14794, 12651,  9285, 13347,  2184, 11302,  5552, 14048,
        12850,  9175,   678, 16158, 12485, 10723, 15296,  6512,  2693,
        11930, 15670,  6224,  4940,   309,  8553, 13950, 10895, 14089,
         3209,  2641,  9406,  8357, 14182, 12018,  3380, 16047,  2742,
        12331]),
 array([746,  66,  63, 879,  98,  22, 943, 323,  46, 342, 273, 114,  73,
        886, 526, 371, 708, 286, 247, 272, 116, 865, 472, 216,  82, 616,
         28, 345, 168, 174, 535, 937,  65, 817, 803, 221

In [80]:
len(requests)

987

In [81]:
requests[0][1]

{6: array([-0.211147  ,  0.028769  , -0.242279  ,  0.0841654 ,  0.0248732 ,
         0.0154993 ,  0.0717655 ,  0.0942284 , -0.00700669, -0.0135621 ,
         0.0214947 ,  0.022302  ,  0.252389  , -0.0336773 , -0.138279  ,
        -0.0534052 ,  0.0690308 , -0.197792  , -0.109553  , -0.00141407,
         0.0443739 ,  0.172978  , -0.00064369,  0.0590741 , -0.0824093 ,
        -0.0682868 ,  0.0344061 ,  0.0911272 , -0.0051644 , -0.0839934 ,
         0.00860636, -0.0956932 , -0.0486178 , -0.0878616 , -0.00121456,
        -0.0839231 , -0.0282804 ,  0.00692664,  0.102235  , -0.0945008 ,
         0.0576525 ,  0.0920146 ,  0.140158  , -0.0620869 , -0.20815   ,
         0.00623474,  0.105515  ,  0.215129  ,  0.0282423 , -0.0290657 ,
        -0.0268382 ,  0.0365566 , -0.0845465 ,  0.0747499 , -0.0457398 ,
        -0.0335554 , -0.0625323 , -0.11611   , -0.00558053,  0.0719073 ,
        -0.114699  ,  0.0206136 , -0.0253669 , -0.0439999 ,  0.326458  ,
        -0.0629797 , -0.0288919 , -0.0863563 , -

In [82]:
embed_users = np.array([
    np.array([r_i[2][i][1] for i in key_games])
    for r_i in requests
])
embed_users_mean = embed_users.mean(axis = 0)
# embed_users = embed_users - embed_users_mean
embed_users.shape

(987, 100)

In [83]:
embed_games = np.array([
    np.array([requests[r_i][2][g_i][1] for r_i in key_reqs])
    for g_i in range(games_count)
])
embed_games_mean = embed_games.mean(axis = 0)
# embed_games = embed_games - embed_games_mean
embed_games.shape

(16514, 101)

In [84]:
games_top = embed_games.mean(axis = 1)

In [85]:
games2users = np.array([
    embed_games[g_i]
    for g_i in key_games
])
games2users.shape

(100, 101)

In [86]:
core_users_scores = np.array([
    np.array([g_i[1] for g_i in requests[r_i][2]])
    for r_i in key_reqs
])
core_users_scores.shape

(101, 16514)

In [87]:
embed_users.shape

(987, 100)

In [88]:
core_users_embs = embed_users[key_reqs]
core_users_embs.shape

(101, 100)

In [89]:
ge_users = (embed_users.T / embed_users.mean(axis=1)).T @ games2users
# ge_users = embed_users @ games2users
ge_users.shape

(987, 101)

In [90]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(500):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 5000
        )
        print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.847605611206494, shape=(), dtype=float64)
1.9079946
1.8526998
1.8164471
1.8170004
1.8069254
1.7839754
1.7684026
1.7640418
1.7634403
1.7613382
1.7563657
1.7498398
1.7441542
1.7411373
1.7403915
1.7392875
1.7362894
1.7325269
1.7297732
1.7282779
1.7270452
1.7254838
1.7237501
1.7219975
1.7201636
1.7184062
1.717088
1.7162386
1.7155193
1.7146997
1.7137927
1.7128273
1.7118518
1.7110538
1.7105333
1.7100797
1.7094367
1.7086526
1.7079452
1.7073987
1.7069688
1.7066064
1.7062389
1.7057991
1.7053434
1.7049927
1.7047404
1.7044779
1.7041705
1.703864
1.7035831
1.7033322
1.7031128
1.7029057
1.7026926
1.7024918
1.7023106
1.7021264
1.7019467
1.7018042
1.701686
1.7015527
1.7014009
1.7012595
1.7011476
1.701059
1.7009655
1.7008494
1.7007331
1.7006454
1.7005782
1.7005055
1.7004238
1.7003465
1.7002804
1.7002237
1.7001674
1.7001078
1.7000513
1.7000006
1.6999519
1.6999053
1.6998627
1.6998214
1.6997817
1.6997454
1.6997112
1.6996778
1.6996458
1.6996152
1.6995876
1.699562
1.6995358
1.69951
1.6

In [91]:
tf.reduce_mean(W * W) * 1

<tf.Tensor: shape=(), dtype=float32, numpy=8.426843e-06>

In [92]:
from scipy.spatial.distance import cosine

from sklearn.metrics.pairwise import cosine_distances, euclidean_distances

from sklearn.metrics import pairwise 

In [93]:
dir(pairwise)

['DataConversionWarning',
 'KERNEL_PARAMS',
 'PAIRED_DISTANCES',
 'PAIRWISE_BOOLEAN_FUNCTIONS',
 'PAIRWISE_DISTANCE_FUNCTIONS',
 'PAIRWISE_KERNEL_FUNCTIONS',
 'Parallel',
 '_NAN_METRICS',
 '_VALID_METRICS',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 '_argmin_min_reduce',
 '_check_chunk_size',
 '_chi2_kernel_fast',
 '_deprecate_positional_args',
 '_dist_wrapper',
 '_euclidean_distances_upcast',
 '_get_mask',
 '_num_samples',
 '_pairwise_callable',
 '_parallel_pairwise',
 '_precompute_metric_params',
 '_return_float_dtype',
 '_sparse_manhattan',
 'additive_chi2_kernel',
 'check_array',
 'check_non_negative',
 'check_paired_arrays',
 'check_pairwise_arrays',
 'chi2_kernel',
 'cosine_distances',
 'cosine_similarity',
 'csr_matrix',
 'delayed',
 'distance',
 'distance_metrics',
 'effective_n_jobs',
 'euclidean_distances',
 'gen_batches',
 'gen_even_slices',
 'get_chunk_n_rows',
 'haversine_distances',
 'is_scalar_nan',


In [94]:
import matplotlib.pyplot as plt
plt.hist(embed_games.flatten())

(array([1.66786e+06, 3.00000e+01, 8.00000e+00, 5.00000e+00, 3.00000e+00,
        4.00000e+00, 2.00000e+00, 1.00000e+00, 0.00000e+00, 1.00000e+00]),
 array([8.88511713e-05, 6.81958441e+01, 1.36391599e+02, 2.04587355e+02,
        2.72783110e+02, 3.40978865e+02, 4.09174621e+02, 4.77370376e+02,
        5.45566131e+02, 6.13761886e+02, 6.81957642e+02]),
 <a list of 10 Patch objects>)

In [ ]:
1. Расстояние
2. embed_users @ W -> F
3. Как выбирать ключевые элементы = ?
4. Сравниться с продовыми нейронками <--
5. Сравнится с ALS

Эксперименты в духе коллег: RPG?
    - см 4 + считаем #вычислений катбуста + приближение топа катбуста
    - полносвязная нейронка
    - АБ
    
    


99. Профит в проде

In [97]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.46297968397291195 0.005586907449209933 0.4553837471783296
d -> 0.47415349887133185 0.006106094808126411 0.4553837471783296
c -> 0.037042889390519196 0.005869074492099322 0.4553837471783296
w -> 0.5063544018058691 0.00636568848758465 0.4553837471783296


In [98]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3709706546275394 0.003115124153498871 0.32559819413092556
d -> 0.3656884875846501 0.002979683972911964 0.32559819413092556
c -> 0.022573363431151242 0.0030925507900677203 0.32559819413092556
w -> 0.39376975169300227 0.003295711060948081 0.32559819413092556


Тюним

In [103]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T) ** 2)
            + tf.reduce_mean(W * W) * 1000
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.847605611206494, shape=(), dtype=float64)
1.885251
1.6313542
1.6180863
1.6158309
1.6154783
1.6154329
1.615451
1.6154275
1.6154274
1.6154281


In [104]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.3709706546275394 0.00363431151241535 0.32559819413092556
d -> 0.3656884875846501 0.00327313769751693 0.32559819413092556
c -> 0.022573363431151242 0.0032054176072234763 0.32559819413092556
w -> 0.4158690744920993 0.0026862302483069977 0.32559819413092556


In [105]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)

topsize = 100
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

e -> 0.46297968397291195 0.005778781038374718 0.4553837471783296
d -> 0.47415349887133185 0.006185101580135441 0.4553837471783296
c -> 0.037042889390519196 0.006422121896162529 0.4553837471783296
w -> 0.5173814898419864 0.006027088036117382 0.4553837471783296


# dssm?

In [140]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-docblocks[i] @ requests_i[1][i] for requests_i in requests], axis=1)
    for i in (6, 7, 8, 9)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"), (rank_di[7], "7"), (rank_di[8], "8"), (rank_di[9], "9"), (rank_e, "e"), (rank_d, "d"), (rank_c, "c"), (rank_w, "w")):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.14814898419864558 0.003115124153498871 0.32559819413092556
7 -> 0.08559819413092551 0.002663656884875847 0.32559819413092556
8 -> 0.142686230248307 0.0029345372460496616 0.32559819413092556
9 -> 0.022753950338600455 0.0029796839729119636 0.32559819413092556
e -> 0.3709706546275394 0.00327313769751693 0.32559819413092556
d -> 0.3656884875846501 0.0029345372460496616 0.32559819413092556
c -> 0.022573363431151242 0.002573363431151242 0.32559819413092556
w -> 0.4158690744920993 0.002979683972911964 0.32559819413092556


In [143]:
games_top

array([0.00511785, 0.00491747, 0.00386624, ..., 0.14714591, 0.05507636,
       0.09588517])

In [153]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-2 * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
    for i in (6,)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

dssmid = 6

for rank, v in ((rank_di[6], "6"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

6 -> 0.3482392776523702 0.00327313769751693 0.32559819413092556


In [155]:
rank_e = np.argsort(euclidean_distances(ge_users, embed_games), axis=1)
rank_d = np.argsort(-ge_users @ embed_games.T, axis=1)
rank_c = np.argsort(cosine_distances(ge_users, embed_games), axis=1)
rank_w = np.argsort(-embed_users @ W @ embed_games.T, axis=1)
rank_di = {
    i: np.argsort([-5 * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
    for i in (7,)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_di[7], "7"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

7 -> 0.38038374717832957 0.0030925507900677203 0.32559819413092556


In [157]:
rank_di = {
    i: np.argsort([-10 * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
    for i in (7,)
}


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_di[7], "7"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids
            
            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

7 -> 0.4094356659142212 0.0032505643340857786 0.32559819413092556


In [159]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (7,)
    }


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[7], "7"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
7 -> 0.32559819413092556 0.0027990970654627545 0.32559819413092556
x =  1
7 -> 0.33426636568848755 0.002889390519187359 0.32559819413092556
x =  2
7 -> 0.3428893905191874 0.002595936794582393 0.32559819413092556
x =  3
7 -> 0.3554401805869074 0.0024830699774266367 0.32559819413092556
x =  4
7 -> 0.3681489841986456 0.0031376975169300227 0.32559819413092556
x =  5
7 -> 0.38038374717832957 0.0032505643340857786 0.32559819413092556
x =  6
7 -> 0.3904740406320542 0.0031376975169300227 0.32559819413092556
x =  7
7 -> 0.4014221218961625 0.0031376975169300227 0.32559819413092556
x =  8
7 -> 0.40925507900677205 0.0030248306997742664 0.32559819413092556
x =  9
7 -> 0.4111738148984199 0.0032731376975169302 0.32559819413092556
x =  10
7 -> 0.4094356659142212 0.0030925507900677203 0.32559819413092556
x =  11
7 -> 0.4055530474040633 0.003250564334085779 0.32559819413092556
x =  12
7 -> 0.3979909706546275 0.003363431151241535 0.32559819413092556
x =  13
7 -> 0.39020316027088037 0.0032505643340

In [160]:
for x in range(20):
    print("x = ", x)
    rank_di = {
        i: np.argsort([-x * docblocks[i] @ requests_i[1][i]-games_top for requests_i in requests], axis=1)
        for i in (6,)
    }


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_di[6], "6"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
6 -> 0.32559819413092556 0.003431151241534989 0.32559819413092556
x =  1
6 -> 0.3330474040632055 0.003295711060948081 0.32559819413092556
x =  2
6 -> 0.3482392776523702 0.0031376975169300227 0.32559819413092556
x =  3
6 -> 0.3683747178329571 0.0027765237020316025 0.32559819413092556
x =  4
6 -> 0.3923702031602709 0.0026636568848758466 0.32559819413092556
x =  5
6 -> 0.4159142212189616 0.002934537246049662 0.32559819413092556
x =  6
6 -> 0.434762979683973 0.0030925507900677203 0.32559819413092556
x =  7
6 -> 0.4462076749435666 0.002866817155756208 0.32559819413092556
x =  8
6 -> 0.44950338600451467 0.002957110609480813 0.32559819413092556
x =  9
6 -> 0.4487584650112867 0.00327313769751693 0.32559819413092556
x =  10
6 -> 0.4443792325056434 0.003318284424379233 0.32559819413092556
x =  11
6 -> 0.43846501128668175 0.002979683972911964 0.32559819413092556
x =  12
6 -> 0.42975169300225735 0.002889390519187359 0.32559819413092556
x =  13
6 -> 0.42130925507900674 0.002415349887133183 0

In [165]:
for x in range(20):
    print("x = ", x)
    rank_w = np.argsort(-x*embed_users @ W @ embed_games.T-games_top, axis=1)


    topsize = 50
    games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

    for rank, v in ((rank_w, "w"),):

        rec_res  = list()
        rand_res = list()
        top_res = list()
        for req_id in range(len(requests)):
            if req_id not in key_reqs:
                rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
                real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
                random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
                top = games_top_ids

                ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

                rec_res.append(ev(rec, real) / float(topsize))
                rand_res.append(ev(random, real) / float(topsize))
                top_res.append(ev(top, real) / float(topsize))

        print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

x =  0
w -> 0.32559819413092556 0.002957110609480813 0.32559819413092556
x =  1
w -> 0.3730925507900677 0.003002257336343115 0.32559819413092556
x =  2
w -> 0.39049661399548535 0.0030699774266365696 0.32559819413092556
x =  3
w -> 0.39970654627539504 0.0025959367945823926 0.32559819413092556
x =  4
w -> 0.40446952595936797 0.0032054176072234763 0.32559819413092556
x =  5
w -> 0.4063656884875847 0.00327313769751693 0.32559819413092556
x =  6
w -> 0.4080361173814898 0.0027088036117381494 0.32559819413092556
x =  7
w -> 0.4083972911963883 0.0032054176072234763 0.32559819413092556
x =  8
w -> 0.40889390519187363 0.003047404063205417 0.32559819413092556
x =  9
w -> 0.40941309255079006 0.0024830699774266367 0.32559819413092556
x =  10
w -> 0.4103837471783296 0.0035214446952595937 0.32559819413092556
x =  11
w -> 0.4109932279909707 0.0032279909706546274 0.32559819413092556
x =  12
w -> 0.41173814898419875 0.0031376975169300227 0.32559819413092556
x =  13
w -> 0.41164785553047406 0.00313769751

In [187]:
print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
print("0-loss = ", tf.reduce_mean((core_users_scores - 0) ** 2))

0-loss =  tf.Tensor(1.7728486536113712, shape=(), dtype=float64)
0-loss =  tf.Tensor(1.847605611206494, shape=(), dtype=float64)


In [188]:
import tensorflow as tf
initializer = tf.keras.initializers.GlorotUniform()
values = initializer(shape=games2users.shape)
W = tf.Variable(values / 100., trainable = True) 
W, W.dtype

print("0-loss = ", tf.reduce_mean((core_users_scores - games_top) ** 2))
loss = tf.keras.losses.MeanSquaredError()
opt =  tf.keras.optimizers.Adam(learning_rate=0.001)

for i in range(1000):
    def loss():
        v = (
            tf.reduce_mean((core_users_scores - core_users_embs @ W @ embed_games.T-games_top) ** 2)
            + tf.reduce_mean(W * W) * 150
        )
        if i % 100 == 0:
            print(v.numpy())
        return v
    
    opt.minimize(loss, [W]).numpy()

0-loss =  tf.Tensor(1.7728486536113712, shape=(), dtype=float64)
1.810549
1.5524985
1.5095793
1.492274
1.484134
1.4800961
1.4780236
1.4769114
1.4762833
1.4774071


In [189]:
rank_w = np.argsort(-embed_users @ W @ embed_games.T-games_top, axis=1)


topsize = 50
games_top_ids = [requests[0][2][g_i][0] for g_i in np.argsort(-games_top)[:topsize]]

for rank, v in ((rank_w, "w"),):

    rec_res  = list()
    rand_res = list()
    top_res = list()
    for req_id in range(len(requests)):
        if req_id not in key_reqs:
            rec = [requests[req_id][2][rank[req_id][i]][0] for i in range(topsize)]
            real = [x[1] for x in sorted([(-r[1], r[0]) for r in requests[req_id][2]])[:topsize]]
            random = np.random.choice([r_i[0] for r_i in requests[0][2]], topsize, replace=False)
            top = games_top_ids

            ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 

            rec_res.append(ev(rec, real) / float(topsize))
            rand_res.append(ev(random, real) / float(topsize))
            top_res.append(ev(top, real) / float(topsize))

    print(v, "->", np.mean(rec_res), np.mean(rand_res), np.mean(top_res))

w -> 0.41433408577878106 0.003047404063205418 0.32559819413092556


In [ ]:
500 -> .4108
250 -> .4117
150 -> .4143
100 -> .4151
050 -> .4142
010 -> .4115
000 -> .4033